# Evaluating Thyroid Disease: Choosing the Right Metric

In medical machine learning, selecting the correct evaluation metric is just as critical as choosing the right model architecture. This dataset presents a classic, severe challenge: **extreme class imbalance**.

Let's start from scratch and understand how to mathematically frame our priorities for detecting Thyroid disease.


## Why Accuracy is Misleading

In our dataset, approximately **90% of patients are "negative"** (healthy), while only 10% fall into the "hyperthyroid" or "hypothyroid" classes. 

If we use standard Accuracy to evaluate our models, we run into the **"Naive Classifier" problem**.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

# Imagine a test set of 1000 patients
# 900 negative (0), 50 hyperthyroid (1), 50 hypothyroid (2)
y_true = np.array([0]*900 + [1]*50 + [2]*50)

# A naive model that predicts EVERYONE is healthy ("negative")
y_pred_naive = np.zeros(1000)

accuracy = accuracy_score(y_true, y_pred_naive)
print(f"Naive Model Accuracy: {accuracy * 100:.2f}%")


Naive Model Accuracy: 90.00%


This model achieves 90% accuracy but is **clinically useless** because it completely failed to detect a single sick patient. Accuracy rewards the model simply for recognizing the majority distribution.


## Clinical Priorities: The Cost of Mistakes

To move past accuracy, we must weigh the real-world cost of different errors:

1. **False Negative (Missed Disease):** A patient has a thyroid condition but the model predicts "negative". They go untreated, which can lead to severe, potentially fatal health complications. **Cost: Extremely High**.
2. **False Positive (False Alarm):** A healthy patient is flagged as sick. In this domain, this simply means the end-user physician orders a secondary blood test which comes back normal. **Cost: Low** (relatively speaking).

Because our primary goal is to minimize False Negatives, our priority becomes **Recall (Sensitivity)**. Recall measures: *"Of all the patients who actually have the disease, how many did we successfully find?"*


## The Pure Recall Approach

Recognizing the need for high sensitivity, we might consider a custom metric like **Thyroid Mean Recall**, defined as the average recall of the two disease classes:

$$\text{Score} = \frac{\text{Recall}_{\text{hyperthyroid}} + \text{Recall}_{\text{hypothyroid}}}{2}$$

This metric heavily prioritizes finding sick patients. **However**, it introduces a mathematical trap known as the **"Trivial Model Problem"**. If a scorer *only* considers Recall, an optimization process (like `GridSearchCV`) will inevitably exploit it by ignoring Precision entirely.


In [ ]:
from sklearn.metrics import recall_score

# The "Trivial Panic Model" - predicts everyone is sick!
# Let's say it flips a coin to predict either hyperthyroid (1) or hypothyroid (2) for EVERY patient
np.random.seed(42)
y_pred_panic = np.random.choice([1, 2], size=1000)

# Calculate Mean Thyroid Recall for this conceptually broken model
hyper_recall = recall_score(y_true, y_pred_panic, labels=[1], average=None)[0]
hypo_recall = recall_score(y_true, y_pred_panic, labels=[2], average=None)[0]

mean_recall = (hyper_recall + hypo_recall) / 2
print(f"Hyperthyroid Recall: {hyper_recall * 100:.2f}%")
print(f"Hypothyroid Recall: {hypo_recall * 100:.2f}%")
print(f"Thyroid Mean Recall Score: {mean_recall * 100:.2f}%")


Hyperthyroid Recall: 54.00%
Hypothyroid Recall: 60.00%
Thyroid Mean Recall Score: 57.00%


Notice that our "Panic" model achieves an artificially inflated ~50% Mean Recall. Mathematically, an ignorant model that blindly predicts "sick" for everyone will always average a 50% Mean Recall (a probability $p$ of guessing hyperthyroid and $1-p$ of guessing hypothyroid averages to $\frac{p + (1 - p)}{2} = 0.5$). It achieves this "free" 50% baseline without learning anything whatsoever, simply by misclassifying **all 900 healthy patients**. 

While False Positives are "cheaper", they are not free. A metric based *purely* on recall encourages an optimizer to completely ignore the healthy majority class to get that guaranteed baseline. But a model with near 0% precision provides zero screening value, because you would have to run expensive follow-up tests on the entire healthy population anyway. **We cannot ignore Precision entirely.**


## The Solution: Balancing Trade-offs with $F_\beta$

We need a metric that aggressively prioritizes Recall without letting Precision drop to zero. The standard $F_1$-Score is the harmonic mean of Precision and Recall, but it weights them equally.

For medical screening, we use the **$F_\beta$ Score**, which explicitly weights Recall as $\beta$ times more important than Precision. By using a **Macro $F_2$ Score** over only our two disease classes, we mathematical enforce that finding a sick patient is twice as important as avoiding a false positive.


In [ ]:
from sklearn.metrics import fbeta_score

# Comparing our models with the robust Macro F2 score targeting ONLY the disease classes (1 and 2)
def print_f2(model_name, y_predictions):
    f2 = fbeta_score(y_true, y_predictions, beta=2, labels=[1, 2], average='macro', zero_division=0)
    print(f"{model_name} Disease F2 Score: {f2:.4f}")

print_f2("Naive Model", y_pred_naive)
print_f2("Panic Model", y_pred_panic)

# A good model that catches 90% of sicknesses with 50% precision
y_pred_good = y_true.copy()
# Add some false positives (50 healthy people predicted as sick)
y_pred_good[0:25] = 1
y_pred_good[25:50] = 2
# It misses a few (5 hyper and 5 hypo predicted as healthy)
y_pred_good[-55:-50] = 0
y_pred_good[-5:-1] = 0

print_f2("Clinically Useful Model", y_pred_good)


Naive Model Disease F2 Score: 0.0000
Panic Model Disease F2 Score: 0.2035
Clinically Useful Model Disease F2 Score: 0.8410


## Implementing the Scorer

To use the F₂ score properly in `cross_val_score` and `RandomizedSearchCV`, we wrap it with `make_scorer`. The resulting `thyroid_scorer` is exported from `src/metrics.py` and used **throughout all subsequent notebooks** as the single evaluation metric for this project.



In [ ]:
from sklearn.metrics import make_scorer

def thyroid_disease_f2_score(y_true, y_pred):
    """
    Macro F2-score targeting strictly the two thyroid disease classes.
    Handles both string labels and integer-encoded labels (alphabetical LabelEncoder ordering:
    hyperthyroid=0, hypothyroid=1, negative=2).
    """
    y_true = np.asarray(y_true)

    if np.issubdtype(y_true.dtype, np.integer) or (len(y_true) > 0 and isinstance(y_true[0], (int, np.integer))):
        h_label, l_label = 0, 1
    else:
        h_label, l_label = 'hyperthyroid', 'hypothyroid'

    return float(fbeta_score(y_true, y_pred, beta=2, labels=[h_label, l_label], average='macro', zero_division=0))

# This scorer is what we import as `thyroid_scorer` from src/metrics.py
thyroid_scorer = make_scorer(thyroid_disease_f2_score)

print("Scorer successfully created:", type(thyroid_scorer))


Scorer successfully created: <class 'sklearn.metrics._scorer._Scorer'>
